In [ ]:
!pip install -q langchain langchain-groq

In [ ]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")   # LangChain reads it from here

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time or place. If it isn't in FACTS,
        say "I'll check with the owner." Work out totals carefully.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),          # the chat so far goes here
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()

history = []
def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))
print(ask("What's my name?"))

Hi Ravi! Yes, we offer egg‑less versions of all our cakes – plum, chocolate, cupcakes and more. Let me know which one you’d like, and I’ll help you with the details!
Your name is Ravi.


In [ ]:
history

[('human', "Hi, I'm Ravi. Do you make eggless cakes?"),
 ('ai',
  'Hi Ravi! Yes, we offer egg‑less versions of all our cakes – plum, chocolate, cupcakes and more. Let me know which one you’d like, and I’ll help you with the details!'),
 ('human', "What's my name?"),
 ('ai', 'Your name is Ravi.')]

In [ ]:
import csv

def run_tests(path: str = "/content/test.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")



In [ ]:
# Let's create a mock test.csv file first so run_tests() has a file to read
import csv

test_data = [
    {"question": "Do you make eggless cakes?", "must_include": "yes"},
    {"question": "How much is a plum cake?", "must_include": "500"},
    {"question": "Can you deliver to my home in 2 hours?", "must_include": "owner"}
]

with open("/content/test.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["question", "must_include"])
    writer.writeheader()
    writer.writerows(test_data)

# Now run the test suite
run_tests()

PASS | Do you make eggless cakes? → Yes! All our cakes—including plum, chocolate and cupcakes—are availabl
PASS | How much is a plum cake? → The plum cake costs ₹500. Let me know if you’d like to add anything el
PASS | Can you deliver to my home in 2 hours? → I’ll check with the owner and get back to you shortly!

3/3 passed (100%)


### Extension: Testing OpenAI and Google Gemini in LangChain
First, we need to install the integration packages for OpenAI and Google GenAI.

In [ ]:
!pip install -q langchain-openai langchain-google-genai

Configure the API keys for all three providers and initialize their respective LangChain Chat Models.

In [ ]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

# Configure all environments with keys from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY") or ""
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") or ""
os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY") or ""

# Initialize the models (Groq, OpenAI, and Gemini)
llm_groq = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3)
llm_openai = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
llm_gemini = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0.3)

### Custom Business Setup: "Sparkle Clean Car Wash"
Let's rewrite the `SYSTEM` prompt for a new business, define 10 custom test questions, write them into `/content/test.csv`, and create a parameterized runner to easily evaluate any of our three LLMs.

In [ ]:
import csv
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

# New Business: Sparkle Clean Car Wash
SYSTEM_BUSINESS = """ROLE:   You are the friendly assistant for Sparkle Clean Car Wash.
FACTS:  Basic Wash Rs 300. Deluxe Polish Rs 800. Interior Detailing Rs 1500.
        Bookings require 2 hours' notice. All washes have an eco-friendly waterless option.
RULES:  Never promise a specific finish time. If it isn't in FACTS,
        say "I'll check with the manager."
FORMAT: Under 50 words, friendly, English."""

# 10 Custom business questions mapped to a 'must_include' requirement
custom_tests = [
    {"question": "Do you have waterless options?", "must_include": "eco"},
    {"question": "How much is the deluxe polish?", "must_include": "800"},
    {"question": "Can you clean my interiors?", "must_include": "1500"},
    {"question": "How much is the basic wash?", "must_include": "300"},
    {"question": "Can you finish polishing my car in exactly 15 minutes?", "must_include": "manager"},
    {"question": "Do you clean engines?", "must_include": "manager"},
    {"question": "Is there any discount?", "must_include": "manager"},
    {"question": "How long does a booking take?", "must_include": "hours"},
    {"question": "What packages do you have?", "must_include": "polish"},
    {"question": "Do you wash bikes?", "must_include": "manager"}
]

with open("/content/custom_tests.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["question", "must_include"])
    writer.writeheader()
    writer.writerows(custom_tests)

def evaluate_model(llm_instance, system_prompt, path="/content/custom_tests.csv"):
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{question}")
    ])
    chain = prompt | llm_instance | StrOutputParser()

    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = chain.invoke({"question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "\u2192", answer.strip().replace('\n', ' ')[:70])
    accuracy = passed / len(rows)
    print(f"\nOverall Accuracy: {passed}/{len(rows)} passed ({accuracy:.0%})\n")
    return accuracy

Now let's run the 10 custom tests across all three providers.

In [ ]:
print("=== TESTING WITH GROQ ===")
try:
    evaluate_model(llm_groq, SYSTEM_BUSINESS)
except Exception as e:
    print(f"Groq Error: {e}\n")

print("=== TESTING WITH OPENAI ===")
try:
    evaluate_model(llm_openai, SYSTEM_BUSINESS)
except Exception as e:
    print(f"OpenAI Error: {e}\n")

print("=== TESTING WITH GEMINI ===")
try:
    evaluate_model(llm_gemini, SYSTEM_BUSINESS)
except Exception as e:
    print(f"Gemini Error: {e}\n")

=== TESTING WITH GROQ ===
PASS | Do you have waterless options? → Yes! All our washes—Basic, Deluxe, and Interior Detailing—offer an eco
PASS | How much is the deluxe polish? → The deluxe polish is Rs 800. Let me know if you’d like to book it!
FAIL | Can you clean my interiors? → Absolutely! Our interior detailing is ₹1,500, and we also offer an eco
PASS | How much is the basic wash? → The basic wash costs Rs 300. 😊
PASS | Can you finish polishing my car in exactly 15 minutes? → I’d love to help! Let me check with the manager about the timing for a
PASS | Do you clean engines? → We offer basic wash, deluxe polish, and interior detailing. Engine cle
PASS | Is there any discount? → I'll check with the manager about any discounts.
FAIL | How long does a booking take? → I’ll check with the manager and get back to you shortly!
PASS | What packages do you have? → We offer three packages:    - **Basic Wash** – Rs 300   - **Deluxe Pol
PASS | Do you wash bikes? → I’m not sure about bike washes—

### Switch: Changing Temperature to 1.0

In [ ]:
# Initialize a model with high temperature
from langchain_openai import ChatOpenAI

try:
    llm_openai_high_temp = ChatOpenAI(model="gpt-4o-mini", temperature=1.0)

    print("--- Run 1 (Temp 1.0) ---")
    evaluate_model(llm_openai_high_temp, SYSTEM_BUSINESS)

    print("--- Run 2 (Temp 1.0) ---")
    evaluate_model(llm_openai_high_temp, SYSTEM_BUSINESS)
except Exception as e:
    print(f"Could not run OpenAI high temperature test: {e}")

--- Run 1 (Temp 1.0) ---
Could not run OpenAI high temperature test: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}


### Conceptual analysis & Task categorization

#### 1. Switching Temperature to 1.0 Observation
Changing the temperature to 1.0 introduces significantly more output variance. At temperature 0.3, the responses are deterministic and consistent. At 1.0, the model uses more creative and varied vocabulary which can occasionally cause strict string-matching validations (`must_include`) to pass or fail dynamically between subsequent runs.

#### 2. Classifications for "Sparkle Clean Car Wash":
* **Plain Code:** Calculating the final receipt tax and billing total based on selected static options.
* **One LLM Call:** Sentiment analysis on reviews left by users on the car wash's website.
* **Agent:** An automated scheduler that negotiates appointment slots with customers by querying a live calendar tool, checking employee shifts, and booking the bay.

In [ ]:
from langchain.agents import create_agent

# Fix the variable references to match 'agent_order' and 'result_order'
agent_order = create_agent(model=llm_groq, tools=[], system_prompt=SYSTEM)   # tools come tomorrow
result_order = agent_order.invoke({"messages": [{"role": "user", "content": "Do you make eggless cakes?"}]})
print(result_order["messages"][-1].content)

Yes! All our cakes—including plum, chocolate, and cupcakes—are available in egg‑less versions. Let us know what you’d like, and we’ll get it ready for you.
